# Linha de base do indicador do parceiro

Este notebook mede, para o modelo de produção, a proporção de predições dentro da margem de 10 por cento acordada no TAPI, por componente e por área, com a função canônica de avaliação.

O modelo de produção é o descrito na Seção 4.3 c da documentação e na Seção 19 do notebook de modelagem: sete Random Forest, um por área térmica, treinados sobre o formato longo da área, na versão da base em que a coluna de perdas é reduzida ao perfil. O modelo é reproduzido aqui sem alteração de hiperparâmetro, de partição ou de tratamento de ausências, de modo que os resultados permaneçam comparáveis aos já registrados.

O notebook de modelagem já reporta a proporção dentro da margem por área, calculada sobre as predições empilhadas de todos os componentes. Este notebook acrescenta três leituras que não estavam registradas: a proporção por componente, o viés e as métricas de decisão contra o limite consolidados por área, e o confronto com o erro sobre o limite reportado no notebook de Gradient Boosting.

Apenas a validação cruzada é avaliada por componente. O conjunto de teste reservado já foi consultado, conforme registrado na Seção 4.3 da documentação, e uma leitura adicional por componente constituiria nova consulta. Os valores por área no teste são citados como registrados.

**Pré-requisitos.** Os sete arquivos `dataset_<area>_perfil.csv` e o `limites_ativos.csv` na pasta `colabs_S3`, e a partição `particao_treino_teste.json` na pasta `preparacao_testes`.

## 1. Configuração e leitura dos dados

A localização dos arquivos, a separação entre entradas e alvos e a leitura da partição seguem o notebook de modelagem, e são repetidas aqui para que este notebook rode sozinho. A partição é lida de arquivo e nunca sorteada.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, glob, json
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SUFIXO = "_perfil"
PASTA_DADOS = "colabs_S3"
PASTA_PARTICAO = "preparacao_testes"

BASES = [
    ".", "..", os.path.join("..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
    os.path.join("notebooks", PASTA_PARTICAO),
]

# Mesmos valores do notebook de modelagem
PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "perfil_perdas", "deflector_hole", "material_coil_support", "cfg",
]
COLUNAS_AUXILIARES = ["run_idx"]
MIN_TREINO = 8
MIN_PREDICOES = 5


def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )


def separar(df, prefixos=PREFIXOS_FEATURES):
    """Devolve (X, cols_alvo); run_idx fica fora das duas listas."""
    cols_entrada = [c for c in df.columns if any(c.startswith(p + "_") for p in prefixos)]
    cols_alvo = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo


print("Diretorio atual:", os.path.abspath("."))

In [ ]:
arquivos = sorted(glob.glob(os.path.join(
    os.path.dirname(resolve(PASTA_DADOS, f"dataset_mainback{SUFIXO}.csv")),
    f"dataset_*{SUFIXO}.csv",
)))
datasets = {}
for arq in arquivos:
    nome = os.path.basename(arq).replace("dataset_", "").replace(SUFIXO + ".csv", "")
    datasets[nome] = pd.read_csv(arq)

limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")

with open(resolve(PASTA_PARTICAO, "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)

DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})


def limite_de(componente):
    """Limite de seguranca do componente, ou None quando nao houver."""
    valor = limites["limite_num"].get(componente)
    return float(valor) if valor is not None and pd.notna(valor) else None


# Conferencias: o estado precisa ser o mesmo do notebook de modelagem
assert len(datasets) == 7, f"esperadas 7 areas, encontradas {len(datasets)}"
for nome, df in datasets.items():
    assert len(df) == particao["n_ensaios"], f"{nome}: {len(df)} linhas"
    # Sem run_idx, a posicao da linha e o indice do ensaio, como no notebook de modelagem
    if "run_idx" in df.columns:
        assert (df["run_idx"].to_numpy() == np.arange(len(df))).all(), f"{nome}: run_idx fora de ordem"
assert len(particao["run_idx_treino"]) == 46 and len(particao["run_idx_teste"]) == 10
assert len(DOBRAS) == 17

print(f"Areas: {len(datasets)} | ensaios por area: {particao['n_ensaios']}")
print(f"Treino: {len(particao['run_idx_treino'])} | teste reservado: "
      f"{len(particao['run_idx_teste'])} | dobras: {len(DOBRAS)}")
print("Componentes com limite tabelado:", int(limites["limite_num"].notna().sum()))
print("Indice do ensaio:", "run_idx" if all("run_idx" in d.columns for d in datasets.values())
      else "posicao da linha (os arquivos de perfil nao trazem run_idx)")

## 2. Função canônica de avaliação

A célula abaixo é a Seção 1 do notebook `colabs_S3/preparacao_testes/metricas.ipynb`, copiada sem alteração, conforme a instrução da própria célula de definição. Nenhuma métrica é recalculada com código próprio neste notebook.

A proporção dentro da margem é o campo `dentro_10_%`: percentual de predições cujo erro absoluto, dividido pela temperatura medida, não excede `MARGEM_KPI`. Com o limite informado, a função devolve também `falso_aprovado`, sob a regra de reprovação `y >= limite`, que é a regra absoluta; a folga de 10 por cento abaixo do limite, descrita no risco R06, não faz parte da função canônica.

In [ ]:
import os
import json

import numpy as np
import pandas as pd


MARGEM_KPI = 0.10   # margem de erro acordada no TAPI
FATOR_REGRA = 0.90  # regra de projeto: aprovar so 10 por cento abaixo do limite (risco R06)


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo
    linha_regra_C         FATOR_REGRA vezes o limite, linha de aprovacao da regra de projeto
    n_reprova_regra_real  ensaios que atingem a linha da regra na realidade
    falso_aprovado_regra  ensaios que atingem a linha na realidade e ficam abaixo dela
                          na predicao; e a aprovacao falsa descrita no risco R06

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

        # Regra de projeto: a mesma linha, 0,9 vez o limite, vale para o real e o previsto
        linha = FATOR_REGRA * float(limite)
        reprova_regra_real = y_real >= linha
        reprova_regra_prevista = y_previsto >= linha
        resultado.update({
            "linha_regra_C": linha,
            "n_reprova_regra_real": int(reprova_regra_real.sum()),
            "falso_aprovado_regra": int(np.sum(reprova_regra_real & ~reprova_regra_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado",
                     "n_reprova_regra_real", "falso_aprovado_regra"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

A bateria de casos de teste do mesmo notebook é executada sobre a cópia, também sem alteração, para confirmar que a função se comporta aqui como na origem.

In [ ]:
testes = []


def caso(nome, condicao, detalhe=""):
    testes.append({"caso": nome, "resultado": "passou" if condicao else "FALHOU",
                   "detalhe": detalhe})
    return condicao


# 1. Predicao perfeita: todo erro e nulo e a variancia e inteiramente explicada
y = np.array([50.0, 80.0, 120.0, 200.0])
m = avaliar(y, y)
caso("predicao perfeita",
     m["MAE"] == 0 and m["RMSE"] == 0 and m["R2"] == 1 and m["vies"] == 0,
     f"MAE={m['MAE']}, R2={m['R2']}")

# 2. Prever a media do proprio conjunto: por definicao o coeficiente e exatamente zero
m = avaliar(y, np.full_like(y, y.mean()))
caso("previsao pela media do conjunto", np.isclose(m["R2"], 0.0), f"R2={m['R2']:.6f}")

# 3. Exemplo calculado a mao: reais 100 e 200, previstos 110 e 180
m = avaliar([100.0, 200.0], [110.0, 180.0])
caso("exemplo conferido manualmente",
     np.isclose(m["MAE"], 15.0) and np.isclose(m["RMSE"], np.sqrt(250.0)),
     f"MAE={m['MAE']:.2f} (esperado 15,00), RMSE={m['RMSE']:.2f} (esperado 15,81)")

# 4. Alvo constante: o coeficiente e indefinido e deve sair como ausente, nao como zero
m = avaliar([5.0, 5.0, 5.0], [5.0, 6.0, 4.0])
caso("alvo constante devolve valor ausente", np.isnan(m["R2"]), f"R2={m['R2']}")

# 5. Subestimacao sistematica: o vies deve ser negativo e igual em modulo ao erro medio
m = avaliar([100.0, 110.0, 120.0], [90.0, 100.0, 110.0])
caso("subestimacao sistematica",
     np.isclose(m["vies"], -10.0) and np.isclose(m["MAE"], 10.0),
     f"vies={m['vies']:.2f}, MAE={m['MAE']:.2f}")

# 6. Alinhamento por posicao, e nao por indice: duas Series com indices distintos
a = pd.Series([100.0, 200.0], index=[10, 11])
b = pd.Series([110.0, 180.0], index=[0, 1])
m = avaliar(a, b)
caso("alinhamento por posicao", np.isclose(m["MAE"], 15.0), f"MAE={m['MAE']:.2f}")

# 7. Entradas invalidas precisam falhar de forma explicita
for descricao, argumentos in [
    ("dimensoes diferentes", ([1.0, 2.0], [1.0])),
    ("conjunto vazio", ([], [])),
    ("valor ausente em y_real", ([1.0, np.nan], [1.0, 2.0])),
]:
    try:
        avaliar(*argumentos)
        caso(f"recusa: {descricao}", False, "nao levantou excecao")
    except ValueError:
        caso(f"recusa: {descricao}", True)


# 8. Erro percentual: reais 100 e 200, previstos 110 e 180, ambos com 10 por cento
m = avaliar([100.0, 200.0], [110.0, 180.0])
caso("erro percentual conferido manualmente",
     np.isclose(m["MAPE_%"], 10.0) and np.isclose(m["WAPE_%"], 10.0),
     f"MAPE={m['MAPE_%']:.2f}%, WAPE={m['WAPE_%']:.2f}%")

# 9. As duas divergem quando os erros se concentram nas leituras baixas
m = avaliar([10.0, 1000.0], [20.0, 1000.0])
caso("WAPE nao e dominado por alvo pequeno",
     np.isclose(m["MAPE_%"], 50.0) and m["WAPE_%"] < 1.0,
     f"MAPE={m['MAPE_%']:.2f}%, WAPE={m['WAPE_%']:.2f}%")

# 10. Proporcao dentro da margem: dois de tres dentro de 10 por cento
m = avaliar([100.0, 100.0, 100.0], [105.0, 115.0, 100.0])
caso("proporcao dentro da margem", np.isclose(m["dentro_10_%"], 200.0 / 3),
     f"dentro_10={m['dentro_10_%']:.1f}%")

# 11. Alvo nao positivo precisa ser recusado, por indicar escala padronizada
try:
    avaliar([0.0, 100.0], [1.0, 100.0])
    caso("recusa: alvo nao positivo", False, "nao levantou excecao")
except ValueError:
    caso("recusa: alvo nao positivo", True)

# 12. Assimetria do erro percentual: prever baixo e menos penalizado que prever alto
baixo = avaliar([100.0, 200.0], [100.0, 100.0])["MAPE_%"]
alto = avaliar([100.0, 200.0], [200.0, 200.0])["MAPE_%"]
caso("assimetria do erro percentual", baixo < alto,
     f"prever 100 -> {baixo:.1f}%, prever 200 -> {alto:.1f}%, com o mesmo erro em graus")

# 13. Decisao contra o limite: o segundo ensaio reprova de verdade e o modelo aprova
m = avaliar([100.0, 120.0], [105.0, 109.0], limite=110.0)
caso("decisao contra o limite",
     m["n_reprova_real"] == 1 and m["falso_aprovado"] == 1
     and m["falso_reprovado"] == 0 and np.isclose(m["acerto_decisao_%"], 50.0),
     f"acerto={m['acerto_decisao_%']:.0f}%, falso aprovado={m['falso_aprovado']}")

# 14. Sem limite informado, as metricas de decisao nao sao produzidas
m = avaliar([100.0, 120.0], [105.0, 109.0])
caso("metricas de decisao ausentes sem limite", "acerto_decisao_%" not in m)

# 15. Regra de projeto: linha em 0,9 x 110 = 99. Os dois ensaios a atingem; o modelo
# preve o primeiro abaixo dela. Pela regra absoluta o falso aprovado e o segundo.
m = avaliar([100.0, 120.0], [98.0, 109.0], limite=110.0)
caso("aprovacao falsa contra a regra de 0,9 x limite",
     np.isclose(m["linha_regra_C"], 99.0) and m["n_reprova_regra_real"] == 2
     and m["falso_aprovado_regra"] == 1 and m["falso_aprovado"] == 1,
     f"linha={m['linha_regra_C']:.1f}, falso aprovado pela regra={m['falso_aprovado_regra']}")

testes = pd.DataFrame(testes)
print(testes.to_string(index=False))
print()
assert (testes["resultado"] == "passou").all(), "ha caso de teste reprovado"
print(f"{len(testes)} casos de teste aprovados.")

## 3. Reprodução do modelo de produção

As funções e o laço abaixo são os da Seção 19 do notebook de modelagem, sem alteração de parâmetro. Cada área recebe um Random Forest de 300 árvores sobre o formato longo, com o patamar e a escala de cada componente calculados apenas sobre as linhas de treino de cada dobra.

A reprodução é conferida contra os valores por área gravados na Seção 19. Divergência acima da tolerância interrompe o notebook, pois indicaria que os números seguintes não descrevem o modelo registrado.

In [ ]:
AREA_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)


def formato_longo(nome, df):
    """Uma linha por par de ensaio e componente medido."""
    X, cols_alvo = separar(df)
    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        sub = X[y.notna()].copy()
        sub["_run"] = (df["run_idx"][y.notna()].to_numpy()
                       if "run_idx" in df.columns else np.arange(len(df))[y.notna()])
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    return longo[longo["_dobra"].notna()].reset_index(drop=True)


def atributos_do_componente(longo, treino):
    """Patamar e escala de cada componente, medidos so no treino da dobra."""
    g = longo[treino].groupby("_componente")["_alvo"]
    media = g.mean()
    desvio = g.std().fillna(1.0).replace(0, 1.0)
    geral = float(longo.loc[treino, "_alvo"].mean())
    return media, desvio, geral


def matriz(longo, mascara, entradas, media, desvio, geral):
    comp = longo.loc[mascara, "_componente"]
    return np.hstack([
        longo.loc[mascara, entradas].to_numpy(),
        comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
        comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1),
    ])


predicoes_area = []
for nome, df in sorted(datasets.items()):
    longo = formato_longo(nome, df)
    entradas = [c for c in longo.columns if not c.startswith("_")]
    for d in DOBRAS:
        treino = (longo["_dobra"] != d).to_numpy()
        val = (longo["_dobra"] == d).to_numpy()
        if treino.sum() < MIN_TREINO or not val.any():
            continue
        media, desvio, geral = atributos_do_componente(longo, treino)
        X_tr = matriz(longo, treino, entradas, media, desvio, geral)
        X_val = matriz(longo, val, entradas, media, desvio, geral)
        escala = StandardScaler()
        modelo = RandomForestRegressor(**AREA_PARAMS)
        modelo.fit(escala.fit_transform(X_tr), longo.loc[treino, "_alvo"])
        p = modelo.predict(escala.transform(X_val))
        bloco = longo.loc[val, ["_componente", "_run", "_alvo"]].copy()
        bloco.columns = ["componente", "ensaio", "real"]
        bloco["dataset"] = nome
        bloco["previsto"] = p
        predicoes_area.append(bloco)

predicoes_area = pd.concat(predicoes_area, ignore_index=True)
print(f"Componentes cobertos: {predicoes_area['componente'].nunique()}")
print(f"Predicoes de validacao: {len(predicoes_area)}")

In [ ]:
# Valores gravados na Secao 19 do notebook de modelagem (validacao cruzada)
REGISTRADO = pd.DataFrame([
    ("coil_temps",    344, 11.71, 65.41), ("ct_knobs",      270, 3.16, 84.44),
    ("ipc_comp",     1616,  7.30, 70.73), ("ipc_cooling",   179, 4.04, 82.12),
    ("main_vent_sys", 230,  4.68, 79.57), ("mainback",      458, 5.68, 71.83),
    ("oven_hmi",      180,  2.42, 90.00),
], columns=["dataset", "n_reg", "MAE_reg", "dentro_10_reg"])

reproduzido = pd.DataFrame([
    {"dataset": nome, **avaliar(g["real"], g["previsto"])}
    for nome, g in predicoes_area.groupby("dataset")
])
conf = REGISTRADO.merge(reproduzido[["dataset", "n", "MAE", "dentro_10_%"]], on="dataset")
conf["dif_MAE"] = conf["MAE"] - conf["MAE_reg"]
conf["dif_dentro_10"] = conf["dentro_10_%"] - conf["dentro_10_reg"]
print(conf.round(2).to_string(index=False))

TOL_MAE, TOL_PCT = 0.01, 0.01
assert (conf["n"] == conf["n_reg"]).all(), "numero de predicoes diverge do registrado"
assert (conf["dif_MAE"].abs() <= TOL_MAE).all(), "MAE diverge do registrado"
assert (conf["dif_dentro_10"].abs() <= TOL_PCT).all(), "dentro_10_% diverge do registrado"
print("\nReproducao conferida contra a Secao 19 do notebook de modelagem.")

## 4. Proporção dentro da margem por componente e por área

A função canônica é aplicada a cada componente, com o limite tabelado quando existe, e o resultado é consolidado por área pela função `consolidar`, também canônica.

A consolidação devolve duas médias. A micro-média pondera pelo número de predições e, para uma proporção, coincide com a proporção calculada sobre as predições empilhadas da área, que é o valor registrado no notebook de modelagem. A macro-média dá o mesmo peso a cada componente, e a distância entre as duas indica se a área esconde componentes muito piores que o agregado. As métricas de decisão são contagens, e por isso somadas.

In [ ]:
linhas = []
for (nome, col), g in predicoes_area.groupby(["dataset", "componente"]):
    linhas.append({"dataset": nome, "componente": col,
                   **avaliar(g["real"], g["previsto"], limite=limite_de(col))})
por_componente = pd.DataFrame(linhas)

COLUNAS_COMP = ["componente", "n", "dentro_10_%", "MAE", "vies", "MAPE_%", "R2",
                "limite_C", "n_reprova_real", "falso_aprovado", "falso_reprovado"]
for nome, bloco in por_componente.groupby("dataset"):
    print(f"--- {nome} ---")
    print(bloco.sort_values("dentro_10_%")[COLUNAS_COMP].round(2).to_string(index=False))
    print()

In [ ]:
por_area = consolidar(por_componente, grupo="dataset")
abaixo_metade = (por_componente.assign(abaixo=por_componente["dentro_10_%"] < 50)
                 .groupby("dataset")["abaixo"].sum().astype(int))
pior = por_componente.loc[por_componente.groupby("dataset")["dentro_10_%"].idxmin(),
                          ["dataset", "componente", "dentro_10_%"]]

tabela_area = (por_area[["dataset", "componentes", "n_total", "dentro_10_%_micro",
                         "dentro_10_%_macro", "MAE_micro", "vies_micro",
                         "n_reprova_real", "falso_aprovado", "falso_reprovado"]]
               .merge(pior.rename(columns={"componente": "pior_componente",
                                           "dentro_10_%": "dentro_10_pior"}), on="dataset")
               .assign(componentes_abaixo_de_50=lambda t: t["dataset"].map(abaixo_metade))
               .sort_values("dentro_10_%_micro")
               .reset_index(drop=True))

# A micro-media precisa coincidir com a proporcao empilhada da secao anterior
_check = tabela_area.merge(reproduzido[["dataset", "dentro_10_%"]], on="dataset")
assert np.allclose(_check["dentro_10_%_micro"], _check["dentro_10_%"])

print("Linha de base do indicador do parceiro, Random Forest de producao, validacao cruzada:")
print(tabela_area.round(2).to_string(index=False))

total = avaliar(predicoes_area["real"], predicoes_area["previsto"])
print(f"\nTodas as areas: {total['n']} predicoes | dentro de 10%: {total['dentro_10_%']:.1f}% "
      f"| MAE {total['MAE']:.2f} C | vies {total['vies']:+.2f} C")
print(f"Componentes abaixo de 50% dentro da margem: "
      f"{int((por_componente['dentro_10_%'] < 50).sum())} de {len(por_componente)}")

In [ ]:
# Coil Temps em dois grupos, pelos limites distintos de bobinas e placas
coil = por_componente[por_componente["dataset"] == "coil_temps"].copy()
coil["grupo"] = np.where(coil["componente"].str.startswith("Aluminium Plate"),
                         "placas (150 C)", "bobinas (220 C)")
print(consolidar(coil, grupo="grupo")[["grupo", "componentes", "n_total",
      "dentro_10_%_micro", "MAE_micro", "vies_micro", "n_reprova_real",
      "falso_aprovado"]].round(2).to_string(index=False))

## 5. Proporção dentro da margem e erro sobre o limite

O notebook `testes_treinamento/treino_gradient_boosting.ipynb` reporta, sob o rótulo de indicador de acuracidade, o `pct_limite`: o erro absoluto médio de cada componente dividido pelo seu limite, promediado entre os componentes. Para o Coil Temps, na versão de perfil, o valor é de 8,09 por cento, abaixo dos 10 por cento, e por isso tem aparência de indicador atendido. O notebook `treino_extratrees_7areas.ipynb` não reporta métrica percentual alguma.

A célula seguinte confronta as duas leituras. Para separar o efeito da métrica do efeito do modelo, o `pct_limite` é também calculado sobre as predições do Random Forest, com a mesma fórmula do notebook de Gradient Boosting. Esse cálculo serve apenas ao confronto e não constitui métrica do projeto.

In [ ]:
# pct_limite registrado no notebook de Gradient Boosting, versao perfil,
# tratamento por mascara (Secao 11 daquele notebook)
PCT_LIMITE_GB = {"coil_temps": 8.09, "ct_knobs": 2.97, "ipc_comp": 8.70,
                 "ipc_cooling": 3.46, "main_vent_sys": 3.51, "mainback": 6.72,
                 "oven_hmi": 3.37}

# Mesma formula do notebook de Gradient Boosting: MAE do componente sobre o
# limite, promediado entre os componentes que tem limite. Apenas para confronto.
com_lim = por_componente[por_componente["limite_C"].notna()].copy()
com_lim["pct_limite"] = 100 * com_lim["MAE"] / com_lim["limite_C"]
pct_limite_rf = com_lim.groupby("dataset")["pct_limite"].mean()

confronto = tabela_area[["dataset", "dentro_10_%_micro"]].copy()
confronto["fora_da_margem_%"] = 100 - confronto["dentro_10_%_micro"]
confronto["MAPE_%_micro"] = confronto["dataset"].map(
    por_area.set_index("dataset")["MAPE_%_micro"])
confronto["pct_limite_RF"] = confronto["dataset"].map(pct_limite_rf)
confronto["pct_limite_GB"] = confronto["dataset"].map(PCT_LIMITE_GB)
confronto["componentes_com_limite"] = confronto["dataset"].map(
    com_lim.groupby("dataset").size()).fillna(0).astype(int)
print(confronto.round(2).to_string(index=False))

print("\nCoil Temps por componente:")
print(com_lim[com_lim["dataset"] == "coil_temps"][
    ["componente", "limite_C", "MAE", "pct_limite", "MAPE_%", "dentro_10_%"]
].sort_values("dentro_10_%").round(2).to_string(index=False))

### Por que as duas leituras divergem

As duas grandezas respondem a perguntas diferentes, e a proximidade entre 8 e 10 por cento é coincidência de escala, não equivalência.

A primeira diferença é de natureza. O `pct_limite` é uma média de magnitudes: calcula o erro médio do componente e só depois o divide. A proporção dentro da margem é uma contagem sobre predições individuais. Uma média abaixo de 10 por cento convive com uma fração relevante de predições fora da margem, e é essa fração que o TAPI descreve. O Coil FR ilustra o ponto: erro sobre o limite de 4,92 por cento e erro percentual médio de 9,82 por cento, ambos abaixo da margem, com apenas 56,52 por cento das predições dentro dela.

A segunda é de denominador. O TAPI fixa a margem sobre a temperatura prevista, e a função canônica divide o erro pela temperatura medida. O `pct_limite` divide pelo limite do componente, que na maior parte das leituras é superior à temperatura medida, uma vez que apenas uma pequena parcela das leituras com limite o atinge. O mesmo erro em graus resulta, portanto, em percentual menor sobre o limite. O efeito é mais forte nas placas de alumínio, cujo limite é de 150 °C e que registram leituras abaixo de 100 °C.

A terceira é de cobertura. O `pct_limite` ignora os componentes sem limite tabelado, e a proporção dentro da margem avalia todos.

A quarta é de modelo. O valor de 8,09 por cento vem do Gradient Boosting com um modelo por componente, e não do Random Forest de produção. Calculado sobre o Random Forest, o `pct_limite` do Coil Temps é de 7,02 por cento, e fica abaixo de 10 por cento nas sete áreas, ao passo que a fração de predições fora da margem varia de 10,00 por cento no Oven HMI a 34,59 por cento no Coil Temps. A divergência vem, portanto, da métrica, e não do algoritmo.

Registra-se ainda uma diferença de regra de decisão. O notebook de Gradient Boosting considera reprovado o componente com temperatura estritamente acima do limite e exclui os componentes marcados como de apenas monitoramento; a função canônica usa temperatura igual ou superior ao limite e não exclui nenhum. As contagens de estouro não previsto daquele notebook e o `falso_aprovado` deste não são, por isso, diretamente comparáveis.

A conclusão é que o `pct_limite` não é leitura do indicador contratado e não deve ser apresentado como tal. A leitura do indicador é a proporção dentro da margem, registrada na Seção 4 por componente e por área.

## 6. Registro

Linha de base do indicador do parceiro para o Random Forest de produção, em validação cruzada leave-one-group-out sobre as 17 configurações de treino, com a função canônica de avaliação. A reprodução do modelo coincide com os valores da Seção 19 do notebook de modelagem em todas as áreas.

| Área | Componentes | Predições | Dentro de 10% | Pior componente | MAE | Viés | Atingem o limite | Falsos aprovados |
|---|---:|---:|---:|---|---:|---:|---:|---:|
| Coil Temps | 8 | 344 | 65,41% | Aluminium Plate FL, 45,00% | 11,71 °C | +0,53 °C | 25 | 19 |
| IPC Comp | 37 | 1.616 | 70,73% | IGBT 11, 58,70% | 7,30 °C | +0,90 °C | 142 | 39 |
| Mainback | 10 | 458 | 71,83% | Convect Motor Winding, 58,70% | 5,68 °C | +0,54 °C | 26 | 16 |
| Main Vent Sys | 5 | 230 | 79,57% | AC Blower Housing, 63,04% | 4,68 °C | +0,31 °C | 0 | 0 |
| IPC Cooling | 4 | 179 | 82,12% | IPC Horizontal Duct Oulet (Inlet IPC Fan), 73,33% | 4,04 °C | +1,06 °C | 0 | 0 |
| CT Knobs | 6 | 270 | 84,44% | Right Encoder, 77,78% | 3,16 °C | +0,56 °C | 0 | 0 |
| Oven HMI | 4 | 180 | 90,00% | A33/Bellatrix Microchip, 88,89% | 2,42 °C | +0,20 °C | 0 | 0 |

Sobre as 3.277 predições, a proporção dentro da margem é de 73,8 por cento. Os 74 falsos aprovados coincidem com o valor sem margem registrado na Seção 4.3 da documentação. O viés e as métricas de decisão seguem a regra absoluta de reprovação da função canônica, e não a folga de 10 por cento abaixo do limite descrita no risco R06.

No Coil Temps, as bobinas têm 74,46 por cento das predições dentro da margem e as placas de alumínio, 55,00 por cento. As placas respondem por 15 dos 19 falsos aprovados da área, e são os únicos componentes do projeto com menos da metade das predições dentro da margem: Aluminium Plate FL e Aluminium Plate RL, ambas com 45,00 por cento.

O TAPI fixa a margem de 10 por cento, mas não a proporção de predições que precisa respeitá-la para que o indicador seja considerado atendido. Essa definição permanece em aberto com a parceira, e sem ela nenhuma área pode ser declarada aprovada ou reprovada no indicador.